<a href="https://colab.research.google.com/github/MrStranger812/MovieLens-CLI-DM/blob/NotebookGCPP/Notebooks/00_Setup_and_Data_Download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
import subprocess
import pkg_resources
from pathlib import Path
from google.colab import drive

def mount_google_drive():
    """Mount Google Drive with proper error handling"""
    try:
        drive.mount('/content/drive', force_remount=True)
        print("✅ Google Drive mounted successfully!")
        return True
    except Exception as e:
        print(f"❌ Error mounting Google Drive: {e}")
        return False

def check_gpu_availability():
    """Check what GPU we have available and configure accordingly"""
    try:
        import torch
        if torch.cuda.is_available():
            gpu_name = torch.cuda.get_device_name(0)
            gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
            print(f"🚀 GPU Detected: {gpu_name}")
            print(f"💾 GPU Memory: {gpu_memory:.1f} GB")
            return True, gpu_name, gpu_memory
        else:
            print("❌ No GPU available")
            return False, None, 0
    except ImportError:
        print("🔧 PyTorch not available, will install")
        return False, None, 0

# Mount Google Drive first
if mount_google_drive():
    # Check initial GPU status
    gpu_available, gpu_name, gpu_memory = check_gpu_availability()

    # Define base path in Google Drive
    GDRIVE_BASE_PATH = Path("/content/drive/MyDrive/Movielens")
    print(f"📁 Project will be saved to: {GDRIVE_BASE_PATH}")
else:
    print("⚠️ Cannot proceed without Google Drive access")
    sys.exit(1)

Mounted at /content/drive
✅ Google Drive mounted successfully!
🚀 GPU Detected: NVIDIA L4
💾 GPU Memory: 22.2 GB
📁 Project will be saved to: /content/drive/MyDrive/Movielens


In [ ]:
import pandas as pd
import numpy as np
import requests
import zipfile
from tqdm import tqdm

def download_with_progress(url, destination):
    """Download file with progress bar"""
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))

    with open(destination, 'wb') as file, tqdm(
        desc=f"Downloading {destination.name}",
        total=total_size,
        unit='iB',
        unit_scale=True,
        unit_divisor=1024,
    ) as progress_bar:
        for chunk in response.iter_content(chunk_size=8192):
            size = file.write(chunk)
            progress_bar.update(size)

def prepare_movielens_data():
    """Download and extract MovieLens 20M dataset to Google Drive"""
    # Create data directories in Google Drive
    raw_data_dir = GDRIVE_BASE_PATH / "data" / "raw"
    raw_data_dir.mkdir(parents=True, exist_ok=True)

    movielens_url = "http://files.grouplens.org/datasets/movielens/ml-20m.zip"
    movielens_zip_path = raw_data_dir / "ml-20m.zip"
    movielens_dir = raw_data_dir / "ml-20m"

    # Download if not exists
    if not movielens_dir.exists():
        print(f"📥 Downloading MovieLens 20M dataset...")
        try:
            download_with_progress(movielens_url, movielens_zip_path)

            print(f"📦 Extracting {movielens_zip_path}...")
            with zipfile.ZipFile(movielens_zip_path, 'r') as zip_ref:
                zip_ref.extractall(raw_data_dir)

            # Clean up zip file to save space
            movielens_zip_path.unlink()
            print("✅ MovieLens 20M dataset ready and zip file cleaned up.")

        except Exception as e:
            print(f"❌ Error downloading/extracting data: {e}")
            return None, None
    else:
        print("✅ MovieLens 20M dataset already available in Google Drive.")

    # Load and preview the data
    ratings_path = movielens_dir / 'ratings.csv'
    movies_path = movielens_dir / 'movies.csv'

    if ratings_path.exists() and movies_path.exists():
        print("📊 Loading ratings and movies data...")

        # Load with chunking for memory efficiency
        ratings_df = pd.read_csv(ratings_path, dtype={
            'userId': 'int32',
            'movieId': 'int32',
            'rating': 'float32',
            'timestamp': 'int64'
        })

        movies_df = pd.read_csv(movies_path, dtype={
            'movieId': 'int32',
            'title': 'str',
            'genres': 'str'
        })

        print("✅ Data loaded successfully.")
        print(f"📈 Ratings shape: {ratings_df.shape}")
        print(f"🎬 Movies shape: {movies_df.shape}")
        print(f"💾 Memory usage: {(ratings_df.memory_usage().sum() + movies_df.memory_usage().sum()) / 1024**2:.1f} MB")

        print("\n📋 Ratings sample:")
        print(ratings_df.head())
        print("\n🎭 Movies sample:")
        print(movies_df.head())

        return ratings_df, movies_df
    else:
        print("❌ Error: MovieLens dataset files not found after extraction.")
        return None, None

# Execute data preparation
ratings_df, movies_df = prepare_movielens_data()

📥 Downloading MovieLens 20M dataset...


📦 Extracting /content/drive/MyDrive/Movielens/data/raw/ml-20m.zip...
✅ MovieLens 20M dataset ready and zip file cleaned up.
📊 Loading ratings and movies data...
✅ Data loaded successfully.
📈 Ratings shape: (20000263, 4)
🎬 Movies shape: (27278, 3)
💾 Memory usage: 382.0 MB

📋 Ratings sample:
   userId  movieId  rating   timestamp
0       1        2     3.5  1112486027
1       1       29     3.5  1112484676
2       1       32     3.5  1112484819
3       1       47     3.5  1112484727
4       1       50     3.5  1112484580

🎭 Movies sample:
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2        

In [ ]:
def create_project_structure():
    """Create comprehensive project structure in Google Drive"""
    directories = [
        'movielens',
        'movielens/preprocessing',
        'movielens/models',
        'movielens/utils',
        'movielens/visualization',
        'data/raw',
        'data/processed',
        'data/interim',
        'data/features',
        'models/trained',
        'models/checkpoints',
        'reports/figures',
        'reports/experiments',
        'notebooks',
        'logs'
    ]

    print("🏗️ Creating project structure...")
    for dir_path in directories:
        full_path = GDRIVE_BASE_PATH / dir_path
        full_path.mkdir(parents=True, exist_ok=True)

    print("✅ Project structure created in Google Drive!")
    return GDRIVE_BASE_PATH

def create_configuration():
    """Create comprehensive configuration file"""
    # Use string concatenation instead of f-string to avoid escaping issues
    config_content = '''"""
Configuration file for MovieLens preprocessing pipeline.
Optimized for Google Colab with Google Drive integration.
"""
from pathlib import Path
import os

# Base directories - Google Drive mounted
BASE_DIR = Path("/content/drive/MyDrive/Movielens")
DATA_DIR = BASE_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
INTERIM_DATA_DIR = DATA_DIR / "interim"
FEATURES_DIR = DATA_DIR / "features"
MODELS_DIR = BASE_DIR / "models"
TRAINED_MODELS_DIR = MODELS_DIR / "trained"
CHECKPOINTS_DIR = MODELS_DIR / "checkpoints"
REPORTS_DIR = BASE_DIR / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
EXPERIMENTS_DIR = REPORTS_DIR / "experiments"
LOGS_DIR = BASE_DIR / "logs"

# Ensure critical directories exist
for dir_path in [DATA_DIR, RAW_DATA_DIR, PROCESSED_DATA_DIR, INTERIM_DATA_DIR,
                 FEATURES_DIR, MODELS_DIR, TRAINED_MODELS_DIR, CHECKPOINTS_DIR,
                 REPORTS_DIR, FIGURES_DIR, EXPERIMENTS_DIR, LOGS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# MovieLens data files
MOVIELENS_DIR = RAW_DATA_DIR / "ml-20m"
RATINGS_FILE = MOVIELENS_DIR / "ratings.csv"
MOVIES_FILE = MOVIELENS_DIR / "movies.csv"
TAGS_FILE = MOVIELENS_DIR / "tags.csv"
LINKS_FILE = MOVIELENS_DIR / "links.csv"

# Processing settings - Optimized for Colab
BATCH_SIZE = 50000  # Reduced for memory efficiency
N_JOBS = 2  # Colab has 2 CPU cores
MEMORY_LIMIT_GB = 10.0  # Conservative limit for Colab
CHUNK_SIZE = 10000  # For chunked processing

# Feature engineering settings
MAX_FEATURES_TFIDF = 100
PCA_VARIANCE_THRESHOLD = 0.95
MIN_RATINGS_PER_USER = 5
MIN_RATINGS_PER_MOVIE = 5

# Model training settings
TRAIN_TEST_SPLIT = 0.8
VALIDATION_SPLIT = 0.1
RANDOM_STATE = 42

# GPU settings (if available)
USE_GPU = ''' + str(gpu_available) + '''
GPU_NAME = "''' + str(gpu_name if gpu_name else 'None') + '''"
GPU_MEMORY_GB = ''' + str(gpu_memory if gpu_memory else 0) + '''

# Logging configuration
LOG_LEVEL = "INFO"
LOG_FORMAT = "%(asctime)s - %(name)s - %(levelname)s - %(message)s"

print("📋 Configuration loaded successfully!")
print(f"🎯 Base directory: {BASE_DIR}")
print(f"🔧 GPU Available: {USE_GPU}")
if USE_GPU:
    print(f"🚀 GPU: {GPU_NAME} ({GPU_MEMORY_GB:.1f} GB)")
'''

    # Save configuration file
    config_path = GDRIVE_BASE_PATH / 'movielens' / 'config.py'
    with open(config_path, 'w') as f:
        f.write(config_content)

    print(f"⚙️ Configuration saved to: {config_path}")
    return config_path

def create_utility_files():
    """Create utility files for the project"""

    # Create __init__.py files
    init_files = [
        'movielens/__init__.py',
        'movielens/preprocessing/__init__.py',
        'movielens/models/__init__.py',
        'movielens/utils/__init__.py',
        'movielens/visualization/__init__.py'
    ]

    for init_file in init_files:
        init_path = GDRIVE_BASE_PATH / init_file
        init_path.touch()

    # Create requirements.txt
    requirements_content = '''# Core data science libraries
pandas>=1.5.0
numpy>=1.24.0
scikit-learn>=1.3.0
matplotlib>=3.7.0
seaborn>=0.12.0
plotly>=5.15.0

# Machine learning
torch>=2.0.0
tensorflow>=2.13.0
lightgbv>=4.0.0
xgboost>=1.7.0

# Utilities
tqdm>=4.65.0
joblib>=1.3.0
requests>=2.31.0
python-dotenv>=1.0.0

# Jupyter specific
ipywidgets>=8.0.0
jupyter>=1.0.0
'''

    requirements_path = GDRIVE_BASE_PATH / 'requirements.txt'
    with open(requirements_path, 'w') as f:
        f.write(requirements_content)

    # Create README.md
    readme_content = '''# MovieLens 20M Recommendation System

This project implements a comprehensive movie recommendation system using the MovieLens 20M dataset.

## Project Structure
```
Movielens/
├── data/
│   ├── raw/           # Original MovieLens data
│   ├── processed/     # Cleaned and processed data
│   ├── interim/       # Intermediate processing results
│   └── features/      # Engineered features
├── movielens/
│   ├── preprocessing/ # Data preprocessing modules
│   ├── models/        # Model implementations
│   ├── utils/         # Utility functions
│   └── visualization/ # Plotting and visualization
├── models/
│   ├── trained/       # Saved trained models
│   └── checkpoints/   # Training checkpoints
├── reports/
│   ├── figures/       # Generated plots and figures
│   └── experiments/   # Experiment results
├── notebooks/         # Jupyter notebooks
└── logs/             # Application logs
```

## Setup
1. Mount Google Drive in Colab
2. Run the setup cells to download data and create structure
3. Install requirements: `pip install -r requirements.txt`

## Usage
Import configuration and start processing:
```python
from movielens.config import *
```
'''

    readme_path = GDRIVE_BASE_PATH / 'README.md'
    with open(readme_path, 'w') as f:
        f.write(readme_content)

    print("📄 Utility files created (requirements.txt, README.md, __init__.py files)")

# Execute project setup
project_base = create_project_structure()
config_path = create_configuration()
create_utility_files()

print(f"🎉 Complete project setup finished!")
print(f"📁 Project location: {GDRIVE_BASE_PATH}")
print(f"⚙️ Configuration: {config_path}")

# Verify the setup
print("\n🔍 Verifying project structure...")
for item in sorted(GDRIVE_BASE_PATH.rglob("*")):
    if item.is_dir():
        print(f"📁 {item.relative_to(GDRIVE_BASE_PATH)}/")
    else:
        print(f"📄 {item.relative_to(GDRIVE_BASE_PATH)}")

print("\n✅ Setup complete! Your MovieLens project is ready in Google Drive.")

🏗️ Creating project structure...
✅ Project structure created in Google Drive!
⚙️ Configuration saved to: /content/drive/MyDrive/Movielens/movielens/config.py
📄 Utility files created (requirements.txt, README.md, __init__.py files)
🎉 Complete project setup finished!
📁 Project location: /content/drive/MyDrive/Movielens
⚙️ Configuration: /content/drive/MyDrive/Movielens/movielens/config.py

🔍 Verifying project structure...
📄 README.md
📁 data/
📁 data/features/
📁 data/interim/
📁 data/ml-20m/
📄 data/ml-20m/README.txt
📄 data/ml-20m/genome-scores.csv
📄 data/ml-20m/genome-tags.csv
📄 data/ml-20m/links.csv
📄 data/ml-20m/movies.csv
📄 data/ml-20m/ratings.csv
📄 data/ml-20m/tags.csv
📄 data/ml-20m.zip
📁 data/processed/
📁 data/raw/
📁 data/raw/ml-20m/
📄 data/raw/ml-20m/README.txt
📄 data/raw/ml-20m/genome-scores.csv
📄 data/raw/ml-20m/genome-tags.csv
📄 data/raw/ml-20m/links.csv
📄 data/raw/ml-20m/movies.csv
📄 data/raw/ml-20m/ratings.csv
📄 data/raw/ml-20m/tags.csv
📁 logs/
📁 models/
📁 models/checkpoints/
📁 m